# 02 Cleaning & Enrichment

Clean the raw data according to project requirements, engineer new features, and export processed files to `data/processed/`.

In [25]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/raw/Bookings.csv')
print(f"Initial shape: {df.shape}")

Initial shape: (103024, 21)


## 1. Drop Junk Columns

In [26]:
junk_cols = ['Vehicle Images', 'Booking_ID', 'Customer_ID']
cols_to_drop = [col for col in junk_cols if col in df.columns]

# Also check for fully null unnamed columns dynamically
unnamed_null_cols = [col for col in df.columns if 'Unnamed' in str(col) and df[col].isnull().all()]
cols_to_drop.extend([c for c in unnamed_null_cols if c not in cols_to_drop])

df = df.drop(columns=cols_to_drop, errors='ignore')

## 2. Fix Datetime

In [27]:
# Parse Date as datetime
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Extract new columns based on Date before modifying it
df['Day_of_Week'] = df['Date'].dt.day_name()
df['Month'] = df['Date'].dt.month_name()

# Extract only the day number from it, rename the column to Day
df['Date'] = df['Date'].dt.day
df = df.rename(columns={'Date': 'Day'})

# Parse Time column to extract Hour
if 'Time' in df.columns:
    df['Time'] = pd.to_datetime(df['Time'], errors='coerce')
    df['Hour'] = df['Time'].dt.hour
    # Drop the redundant Time column
    df = df.drop(columns=['Time'])

# Time_of_Day categorization
def categorize_time_of_day(hour):
    if pd.isna(hour):
        return np.nan
    elif 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df['Time_of_Day'] = df['Hour'].apply(categorize_time_of_day)

## 3. Standardize Categorical Columns

In [28]:
# Strip whitespace from all string columns
string_cols = df.select_dtypes(include=['object']).columns
df[string_cols] = df[string_cols].apply(lambda x: x.str.strip())

# Standardize values
for col in ['Vehicle_Type', 'Pickup_Location', 'Drop_Location']:
    if col in df.columns:
        df[col] = df[col].str.title()

if 'Booking_Status' in df.columns:
    df['Booking_Status'] = df['Booking_Status'].str.title()

## 4. Remove Duplicates

In [29]:
df = df.drop_duplicates()

## 5. Validate Numeric Columns

In [30]:
for col in ['Ride_Distance', 'Booking_Value']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        negative_mask = df[col] < 0
        if negative_mask.any():
            print(f"Anomaly found: {negative_mask.sum()} negative values in {col}")
        else:
            print(f"No negative values found in {col}")

No negative values found in Ride_Distance
No negative values found in Booking_Value


## 6. Add Useful Engineered Columns

In [31]:
df['Is_Peak_Hour'] = df['Hour'].isin([7, 8, 9, 10, 17, 18, 19, 20, 21])
df['Is_Weekend'] = df['Day_of_Week'].isin(['Saturday', 'Sunday'])

if 'Booking_Value' in df.columns and 'Ride_Distance' in df.columns:
    # Handle divide by zero for cancelled rides
    df['Booking_Value_per_KM'] = np.where(df['Ride_Distance'] > 0, df['Booking_Value'] / df['Ride_Distance'], 0)

## 7. Split into Two Dataframes

In [32]:
if 'Booking_Status' in df.columns:
    success_df = df[df['Booking_Status'] == 'Success'].copy()
    cancelled_df = df[df['Booking_Status'] != 'Success'].copy()
else:
    success_df = pd.DataFrame()
    cancelled_df = pd.DataFrame()

## 8. Clean success_df

In [33]:
if not success_df.empty:
    # Fill nulls in Driver_Ratings and Customer_Rating with their median first
    for col in ['Driver_Ratings', 'Customer_Rating']:
        if col in success_df.columns:
            success_df[col] = success_df[col].fillna(success_df[col].median())
    
    # Drop nulls in V_TAT, C_TAT, and Payment_Method
    cols_to_dropna = [col for col in ['V_TAT', 'C_TAT', 'Payment_Method'] if col in success_df.columns]
    success_df = success_df.dropna(subset=cols_to_dropna)
    
    # Remove outliers using the IQR method
    outlier_cols = ['V_TAT', 'C_TAT', 'Ride_Distance', 'Driver_Ratings', 'Customer_Rating']
    for col in outlier_cols:
        if col in success_df.columns:
            success_df[col] = pd.to_numeric(success_df[col], errors='coerce')
            Q1 = success_df[col].quantile(0.25)
            Q3 = success_df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            success_df = success_df[(success_df[col] >= lower_bound) & (success_df[col] <= upper_bound)]
            
    # Fill any remaining nulls in Driver_Ratings and Customer_Rating with their mean
    for col in ['Driver_Ratings', 'Customer_Rating']:
        if col in success_df.columns:
            success_df[col] = success_df[col].fillna(success_df[col].mean())
            
    # Drop fully null columns for successful rides
    cols_to_drop_success = ['Canceled_Rides_by_Customer', 'Canceled_Rides_by_Driver', 'Incomplete_Rides_Reason']
    success_df = success_df.drop(columns=[c for c in cols_to_drop_success if c in success_df.columns])

## 9. Clean cancelled_df

In [34]:
if not cancelled_df.empty:
    # Fill nulls in Canceled_Rides_by_Customer and Canceled_Rides_by_Driver with 'No_Reason'
    for col in ['Canceled_Rides_by_Customer', 'Canceled_Rides_by_Driver']:
        if col in cancelled_df.columns:
            cancelled_df[col] = cancelled_df[col].fillna('No_Reason')
            
    # Fill nulls in Incomplete_Rides and Incomplete_Rides_Reason with 'No_Reason'
    for col in ['Incomplete_Rides', 'Incomplete_Rides_Reason']:
        if col in cancelled_df.columns:
            cancelled_df[col] = cancelled_df[col].fillna('No_Reason')

## 10. Export

In [35]:
import os
os.makedirs('../data/processed', exist_ok=True)
success_df.to_csv('../data/processed/cleaned_dataset.csv', index=False)
cancelled_df.to_csv('../data/processed/cancelled_dataset.csv', index=False)

## 11. Final Verification

In [36]:
print("=== Success DataFrame ===")
print(f"Shape: {success_df.shape}")
print(f"Columns: {list(success_df.columns)}")
print(f"\nNull Counts:\n{success_df.isnull().sum()}\n")

print("=== Cancelled DataFrame ===")
print(f"Shape: {cancelled_df.shape}")
print(f"Columns: {list(cancelled_df.columns)}")
print(f"\nNull Counts:\n{cancelled_df.isnull().sum()}\n")

print("=== Value Counts ===")
for col in ['Vehicle_Type', 'Booking_Status', 'Payment_Method', 'Time_of_Day']:
    if col in df.columns:
        print(f"\n{col} (Combined original df with clean categories):")
        print(df[col].value_counts(dropna=False))

=== Success DataFrame ===
Shape: (63967, 20)
Columns: ['Day', 'Booking_Status', 'Vehicle_Type', 'Pickup_Location', 'Drop_Location', 'V_TAT', 'C_TAT', 'Incomplete_Rides', 'Booking_Value', 'Payment_Method', 'Ride_Distance', 'Driver_Ratings', 'Customer_Rating', 'Day_of_Week', 'Month', 'Hour', 'Time_of_Day', 'Is_Peak_Hour', 'Is_Weekend', 'Booking_Value_per_KM']

Null Counts:
Day                     0
Booking_Status          0
Vehicle_Type            0
Pickup_Location         0
Drop_Location           0
V_TAT                   0
C_TAT                   0
Incomplete_Rides        0
Booking_Value           0
Payment_Method          0
Ride_Distance           0
Driver_Ratings          0
Customer_Rating         0
Day_of_Week             0
Month                   0
Hour                    0
Time_of_Day             0
Is_Peak_Hour            0
Is_Weekend              0
Booking_Value_per_KM    0
dtype: int64

=== Cancelled DataFrame ===
Shape: (39057, 23)
Columns: ['Day', 'Booking_Status', 'Vehicle_T